# Baseline: CoLES via PyTorch-LifeStream

This notebook trains CoLES with self-supervision, then evaluates two classifier-head regimes: frozen encoder + learned head and full fine-tuning. It reports per-seed, mean, and std rows per label fraction.


In [ ]:
# Cell 1 - Setup
import copy
import importlib.util
import json
import pathlib
import random
import subprocess
import sys
import warnings

warnings.filterwarnings("ignore")


def pip_install(*packages):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", *packages, "--quiet"],
        check=True,
    )


subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-e",
        "/kaggle/working/denoising-event-sequences",
        "--quiet",
    ],
    check=True,
)
sys.path.insert(0, "/kaggle/working/denoising-event-sequences")

# Keep the CoLES notebook independent of Lightning. In Kaggle, importing
# pytorch_lightning can touch torch._dynamo and fail on NumPy ABI mismatches.
required_packages = {"pytorch-lifestream": "ptls"}
missing_packages = [
    pkg for pkg, module_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    pip_install(*missing_packages)


def check_numpy_stack():
    return subprocess.run(
        [
            sys.executable,
            "-c",
            "import numpy; import numpy.random; import pandas; import sklearn; import scipy",
        ],
        text=True,
        capture_output=True,
    )


stack_check = check_numpy_stack()
if stack_check.returncode != 0:
    print("Repairing NumPy/pandas/scikit-learn/scipy binary stack...")
    print(stack_check.stderr[-2000:])
    pip_install(
        "--force-reinstall",
        "--no-cache-dir",
        "numpy==1.26.4",
        "pandas==2.2.3",
        "scikit-learn==1.5.2",
        "scipy==1.13.1",
    )
    stack_check = check_numpy_stack()
    if stack_check.returncode != 0:
        raise RuntimeError(
            "Scientific Python stack is still binary-incompatible after reinstall. "
            "Restart the Kaggle session and run the notebook from the first cell again.\n"
            + stack_check.stderr[-2000:]
        )

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader

from ptls.data_load.utils import collate_feature_dict
from ptls.nn import RnnSeqEncoder, TrxEncoder

from src.data.splits import make_entity_splits
from src.utils.seed import set_seed

SMOKE_RUN = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
print("Device:", device)


In [ ]:
# Cell 2 - Paths and config
WORKING_DIR = pathlib.Path("/kaggle/working")
DATA_DIR = WORKING_DIR / "data"
OUTPUT_DIR = WORKING_DIR / "outputs"
CKPT_DIR = OUTPUT_DIR / "checkpoints_coles_ptls"
for p in [OUTPUT_DIR, CKPT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

config = {
    "data": {
        "max_seq_len": 256,
        "min_seq_len": 2,
        "event_type_col": "MCC",
        "timestamp_col": "TRDATETIME",
        "target_col": "target_flag",
        "numerical_cols": ["amount"],
        "categorical_cols": ["channel_type", "trx_category"],
        "group_col": "cl_id",
        "truncation_pretrain": "sliding_window",
        "truncation_eval": "last_events",
        "amount_transform": "robust_scaler",
        "time_transform": "none",
        "train_ratio": 0.70,
        "val_ratio": 0.15,
        "test_ratio": 0.15,
        "min_vocab_count": 5,
    },
    "model": {
        "event_type_emb_dim": 64,
        "cat_emb_dim": 32,
        "num_projection_dim": 64,
        "time_projection_dim": 64,
        "hidden_dim": 256,
        "num_layers": 4,
        "num_heads": 8,
        "dim_feedforward": 1024,
        "dropout": 0.10,
        "activation": "gelu",
        "max_seq_len": 256,
    },
    "pooling": {"type": "gated_attention"},
    "training": {
        "batch_size": 128,
        "num_epochs_finetune": 20,
        "lr": 3e-4,
        "lr_encoder": 3e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.05,
        "gradient_clip_val": 1.0,
        "mixed_precision": USE_AMP,
        "early_stopping_patience": 5,
        "log_every_n_steps": 50,
    },
}

LABEL_FRACTIONS = [0.05, 0.25, 1.00]
LABEL_SAMPLING_SEEDS = [42, 43, 44]
METRIC_COLS = ["accuracy", "macro_f1", "weighted_f1", "roc_auc", "pr_auc", "balanced_accuracy"]

COLES_EPOCHS = 1 if SMOKE_RUN else 10
COLES_FINETUNE_EPOCHS = 2 if SMOKE_RUN else 20
COLES_BATCH_SIZE = 64
EMBEDDING_SIZE = 128
COLES_HEAD_LR = 3e-4
COLES_ENCODER_LR = 3e-5
COLES_WEIGHT_DECAY = 1e-4
COLES_PATIENCE = 2 if SMOKE_RUN else 5
COLES_DOWNSTREAM_MODES = ["frozen_head", "full_finetune"]
print("Output dir:", OUTPUT_DIR)

In [ ]:
# Cell 3 - Load data and shared splits
GROUP_COL = config["data"]["group_col"]
TARGET_COL = config["data"]["target_col"]
TIME_COL = config["data"]["timestamp_col"]
EVENT_COL = config["data"]["event_type_col"]
NUM_COL = "amount"
CAT_COLS = [EVENT_COL] + config["data"]["categorical_cols"]

df = pd.read_csv(DATA_DIR / "rosbank" / "train.csv.gz")
df[TIME_COL] = pd.to_datetime(df[TIME_COL], format="%d%b%y:%H:%M:%S")
labels_by_entity = df.groupby(GROUP_COL)[TARGET_COL].first().to_dict()

splits = make_entity_splits(
    df,
    entity_col=GROUP_COL,
    target_col=TARGET_COL,
    train_ratio=config["data"]["train_ratio"],
    val_ratio=config["data"]["val_ratio"],
    test_ratio=config["data"]["test_ratio"],
    seed=42,
)
all_entity_ids = splits["train"] + splits["val"] + splits["test"]
print(df.shape)
print({k: len(v) for k, v in splits.items()})


In [ ]:
# Cell 4 - Shared sampling, metrics, and aggregation helpers
def get_entity_labels(df, group_col, target_col):
    return df.groupby(group_col)[target_col].first().to_dict()


def sample_labeled_entities(train_ids, labels_by_entity, fraction, seed):
    train_ids = list(train_ids)
    if fraction >= 1.0:
        return train_ids

    rng = np.random.default_rng(seed)
    by_label = {}
    for entity_id in train_ids:
        label = int(labels_by_entity[entity_id])
        by_label.setdefault(label, []).append(entity_id)

    sampled = []
    for label, ids in sorted(by_label.items()):
        ids = np.array(ids, dtype=object)
        n = max(1, int(round(len(ids) * fraction)))
        n = min(n, len(ids))
        sampled.extend(rng.choice(ids, size=n, replace=False).tolist())

    rng.shuffle(sampled)
    return sampled


def compute_binary_metrics(y_true, pos_proba):
    y_true = np.asarray(y_true).astype(int)
    pos_proba = np.asarray(pos_proba).astype(float)
    y_pred = (pos_proba >= 0.5).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, pos_proba)),
        "pr_auc": float(average_precision_score(y_true, pos_proba)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }


def append_mean_std_rows(rows, fractions, metric_cols=METRIC_COLS):
    final_rows = []
    for fraction in fractions:
        runs = [r for r in rows if float(r["fraction"]) == float(fraction)]
        final_rows.extend(runs)
        mean_row = {"fraction": fraction, "seed": "mean", "n_seeds": len(runs)}
        std_row = {"fraction": fraction, "seed": "std", "n_seeds": len(runs)}
        for metric in metric_cols:
            vals = [float(r[metric]) for r in runs if metric in r and pd.notna(r[metric])]
            if vals:
                mean_row[metric] = float(np.mean(vals))
                std_row[metric] = float(np.std(vals, ddof=0))
        final_rows.append(mean_row)
        final_rows.append(std_row)
    return final_rows


def save_results(rows, stem):
    results_df = pd.DataFrame(rows)
    csv_path = OUTPUT_DIR / f"{stem}.csv"
    json_path = OUTPUT_DIR / f"{stem}.json"
    results_df.to_csv(csv_path, index=False)
    json_path.write_text(json.dumps(rows, indent=2, ensure_ascii=False))
    print(f"Saved: {csv_path}")
    print(f"Saved: {json_path}")
    return results_df


In [ ]:
# Cell 5 - PTLS preprocessing fitted on train only
def fit_category_maps(df_train, cat_cols):
    maps = {}
    for col in cat_cols:
        values = df_train[col].astype(str).value_counts().index.tolist()
        mapping = {"<UNK>": 1}
        for value in values:
            if value not in mapping:
                mapping[value] = len(mapping) + 1
        maps[col] = mapping
    return maps


def encode_categories(df_source, maps):
    out = df_source.copy()
    for col, mapping in maps.items():
        out[col] = out[col].astype(str).map(lambda x, m=mapping: m.get(x, 1)).astype("int64")
    return out


def fit_amount_scaler(df_train):
    values = pd.to_numeric(df_train[NUM_COL], errors="coerce").fillna(0.0).values.astype(float)
    center = float(np.median(values))
    q75, q25 = np.percentile(values, [75, 25])
    scale = float(q75 - q25) or 1.0
    return center, scale


def make_ptls_records(df_encoded, entity_ids, include_target=False):
    records = []
    entity_set = set(entity_ids)
    subset = df_encoded[df_encoded[GROUP_COL].isin(entity_set)].sort_values([GROUP_COL, TIME_COL], kind="stable")
    for entity_id, ent in subset.groupby(GROUP_COL, sort=False):
        rec = {
            "event_time": torch.tensor(ent["event_time"].values, dtype=torch.float32),
            EVENT_COL: torch.tensor(ent[EVENT_COL].values, dtype=torch.long),
            "channel_type": torch.tensor(ent["channel_type"].values, dtype=torch.long),
            "trx_category": torch.tensor(ent["trx_category"].values, dtype=torch.long),
            NUM_COL: torch.tensor(ent[NUM_COL].values, dtype=torch.float32),
        }
        if include_target:
            rec["target"] = torch.tensor(int(labels_by_entity[entity_id]), dtype=torch.long)
        records.append((entity_id, rec))
    by_id = {entity_id: rec for entity_id, rec in records}
    return [by_id[entity_id] for entity_id in entity_ids if entity_id in by_id]

train_df = df[df[GROUP_COL].isin(set(splits["train"]))]
category_maps = fit_category_maps(train_df, CAT_COLS)
amount_center, amount_scale = fit_amount_scaler(train_df)

df_ptls = encode_categories(df, category_maps)
df_ptls[NUM_COL] = pd.to_numeric(df_ptls[NUM_COL], errors="coerce").fillna(amount_center)
df_ptls[NUM_COL] = (df_ptls[NUM_COL] - amount_center) / amount_scale
df_ptls["event_time"] = df_ptls[TIME_COL].astype("int64") // 10**9

train_records = make_ptls_records(df_ptls, splits["train"])
val_records = make_ptls_records(df_ptls, splits["val"])
test_records = make_ptls_records(df_ptls, splits["test"])
all_records = make_ptls_records(df_ptls, all_entity_ids)

embedding_specs = {
    col: {"in": max(mapping.values()) + 1, "out": 32 if col == EVENT_COL else 16}
    for col, mapping in category_maps.items()
}
print("PTLS records:", len(train_records), len(val_records), len(test_records))
print("Embedding specs:", embedding_specs)


In [ ]:
# Cell 6 - Manual CoLES-style contrastive pretraining
MIN_SLICE_LEN = 10
CONTRASTIVE_TEMPERATURE = 0.10


def seed_all(seed):
    set_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def slice_record(record, start, end):
    return {k: v[start:end] for k, v in record.items()}


def sample_slice(record):
    n = int(record["event_time"].shape[0])
    if n <= 1:
        return slice_record(record, 0, n)
    min_len = min(MIN_SLICE_LEN, n)
    max_len = min(config["data"]["max_seq_len"], n)
    if min_len > max_len:
        min_len = max_len
    length = random.randint(min_len, max_len)
    start = random.randint(0, n - length)
    return slice_record(record, start, start + length)


def collate_coles_views(records):
    views = []
    pair_ids = []
    for pair_id, record in enumerate(records):
        views.append(sample_slice(record))
        views.append(sample_slice(record))
        pair_ids.extend([pair_id, pair_id])
    batch = collate_feature_dict(views)
    batch.payload["pair_id"] = torch.tensor(pair_ids, dtype=torch.long)
    return batch


def batch_to_device(batch, device):
    return batch.to(device) if hasattr(batch, "to") else batch


def contrastive_pair_loss(embeddings, temperature=CONTRASTIVE_TEMPERATURE):
    z = F.normalize(embeddings, dim=-1)
    logits = z @ z.T / temperature
    batch_size = logits.shape[0]
    diag = torch.eye(batch_size, dtype=torch.bool, device=logits.device)
    logits = logits.masked_fill(diag, -1e9)
    targets = torch.arange(batch_size, device=logits.device)
    targets = torch.where(targets % 2 == 0, targets + 1, targets - 1)
    return F.cross_entropy(logits, targets)


seed_all(42)

seq_encoder = RnnSeqEncoder(
    trx_encoder=TrxEncoder(
        embeddings=embedding_specs,
        numeric_values={NUM_COL: "identity"},
    ),
    hidden_size=EMBEDDING_SIZE,
    is_reduce_sequence=True,
).to(device)

train_coles_loader = DataLoader(
    train_records,
    batch_size=COLES_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_coles_views,
    num_workers=0,
)
val_coles_loader = DataLoader(
    val_records,
    batch_size=COLES_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_coles_views,
    num_workers=0,
)

optimizer = torch.optim.AdamW(seq_encoder.parameters(), lr=1e-3, weight_decay=1e-5)
scaler = torch.cuda.amp.GradScaler() if USE_AMP else None
best_state = None
best_val_loss = float("inf")

for epoch in range(COLES_EPOCHS):
    seq_encoder.train()
    train_losses = []
    for batch in train_coles_loader:
        batch = batch_to_device(batch, device)
        optimizer.zero_grad()
        if USE_AMP:
            with torch.autocast(device_type="cuda"):
                emb = seq_encoder(batch)
                loss = contrastive_pair_loss(emb)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            emb = seq_encoder(batch)
            loss = contrastive_pair_loss(emb)
            loss.backward()
            optimizer.step()
        train_losses.append(float(loss.item()))

    seq_encoder.eval()
    val_losses = []
    with torch.no_grad():
        for batch in val_coles_loader:
            batch = batch_to_device(batch, device)
            emb = seq_encoder(batch)
            val_losses.append(float(contrastive_pair_loss(emb).item()))
    train_loss = float(np.mean(train_losses)) if train_losses else float("nan")
    val_loss = float(np.mean(val_losses)) if val_losses else float("nan")
    print(f"CoLES epoch={epoch:02d} train_loss={train_loss:.4f} val_loss={val_loss:.4f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in seq_encoder.state_dict().items()}
        torch.save(
            {"seq_encoder_state_dict": best_state, "val_loss": best_val_loss},
            CKPT_DIR / "coles_seq_encoder.pt",
        )

if best_state is not None:
    seq_encoder.load_state_dict(best_state)
print("Manual CoLES pretraining complete")


In [ ]:
# Cell 7 - Supervised CoLES classifier helpers
train_records_labeled = make_ptls_records(df_ptls, splits["train"], include_target=True)
val_records_labeled = make_ptls_records(df_ptls, splits["val"], include_target=True)
test_records_labeled = make_ptls_records(df_ptls, splits["test"], include_target=True)

def collate_labeled_ptls(records):
    labels = torch.tensor([int(item["target"]) for item in records], dtype=torch.long)
    feature_records = [
        {k: v for k, v in item.items() if k != "target"}
        for item in records
    ]
    batch = collate_feature_dict(feature_records)
    batch.payload["target"] = labels
    return batch


def make_labeled_loader(entity_ids, shuffle):
    records = make_ptls_records(df_ptls, entity_ids, include_target=True)
    return DataLoader(
        records,
        batch_size=COLES_BATCH_SIZE,
        shuffle=shuffle,
        collate_fn=collate_labeled_ptls,
        num_workers=0,
    )

val_loader = DataLoader(
    val_records_labeled,
    batch_size=COLES_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_labeled_ptls,
    num_workers=0,
)
test_loader = DataLoader(
    test_records_labeled,
    batch_size=COLES_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_labeled_ptls,
    num_workers=0,
)

class CoLESClassifier(nn.Module):
    def __init__(self, seq_encoder, embedding_size, num_classes=2, dropout=0.10):
        super().__init__()
        self.seq_encoder = seq_encoder
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(embedding_size, num_classes),
        )

    def forward(self, batch):
        emb = self.seq_encoder(batch)
        return self.classifier(emb)


def batch_labels(batch):
    labels = batch.payload["target"]
    if labels.ndim > 1:
        labels = labels.view(labels.shape[0], -1)[:, 0]
    return labels.long()


def set_encoder_trainable(model, trainable):
    for param in model.seq_encoder.parameters():
        param.requires_grad = trainable


def build_coles_classifier(mode):
    seq_encoder_copy = copy.deepcopy(seq_encoder)
    model = CoLESClassifier(seq_encoder_copy, EMBEDDING_SIZE).to(device)
    if mode == "frozen_head":
        set_encoder_trainable(model, False)
    elif mode == "full_finetune":
        set_encoder_trainable(model, True)
    else:
        raise ValueError(f"Unknown CoLES downstream mode: {mode}")
    return model


def build_optimizer(model, mode):
    if mode == "frozen_head":
        params = [p for p in model.classifier.parameters() if p.requires_grad]
        return torch.optim.AdamW(params, lr=COLES_HEAD_LR, weight_decay=COLES_WEIGHT_DECAY)
    return torch.optim.AdamW(
        [
            {"params": model.seq_encoder.parameters(), "lr": COLES_ENCODER_LR},
            {"params": model.classifier.parameters(), "lr": COLES_HEAD_LR},
        ],
        weight_decay=COLES_WEIGHT_DECAY,
    )


def evaluate_coles_classifier(model, loader):
    model.eval()
    all_probs = []
    all_labels = []
    with torch.no_grad():
        for batch in loader:
            batch = batch_to_device(batch, device)
            labels = batch_labels(batch)
            logits = model(batch)
            probs = torch.softmax(logits, dim=-1)[:, 1]
            all_probs.append(probs.detach().cpu().numpy())
            all_labels.append(labels.detach().cpu().numpy())
    y_true = np.concatenate(all_labels)
    y_score = np.concatenate(all_probs)
    metrics = compute_binary_metrics(y_true, y_score)
    pred_df = pd.DataFrame({"target": y_true, "proba": y_score})
    return metrics, pred_df


def train_coles_classifier(mode, train_ids, seed, run_dir):
    set_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    train_loader = make_labeled_loader(train_ids, shuffle=True)
    model = build_coles_classifier(mode)
    optimizer = build_optimizer(model, mode)
    scaler = torch.cuda.amp.GradScaler() if USE_AMP else None

    best_state = None
    best_val_auc = -float("inf")
    patience_counter = 0

    for epoch in range(COLES_FINETUNE_EPOCHS):
        model.train()
        total_loss = 0.0
        total_count = 0
        for batch in train_loader:
            batch = batch_to_device(batch, device)
            labels = batch_labels(batch)
            optimizer.zero_grad()
            if USE_AMP:
                with torch.autocast(device_type="cuda"):
                    logits = model(batch)
                    loss = F.cross_entropy(logits, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(batch)
                loss = F.cross_entropy(logits, labels)
                loss.backward()
                optimizer.step()
            total_loss += float(loss.item()) * int(labels.shape[0])
            total_count += int(labels.shape[0])

        val_metrics, _ = evaluate_coles_classifier(model, val_loader)
        val_auc = val_metrics["roc_auc"]
        train_loss = total_loss / max(1, total_count)
        print(
            f"{mode} epoch={epoch:02d} train_loss={train_loss:.4f} "
            f"val_roc_auc={val_auc:.4f}"
        )
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            patience_counter = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            run_dir.mkdir(parents=True, exist_ok=True)
            torch.save({"model_state_dict": best_state, "val_metrics": val_metrics}, run_dir / "best.pt")
        else:
            patience_counter += 1
            if patience_counter >= COLES_PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val_auc

print("Supervised CoLES loaders ready")


In [ ]:
# Cell 8 - Low-label CoLES classifier-head runs
run_rows = []
prediction_rows = []

for mode in COLES_DOWNSTREAM_MODES:
    baseline_name = f"coles_{mode}"
    for fraction in LABEL_FRACTIONS:
        for seed in LABEL_SAMPLING_SEEDS:
            train_ids = sample_labeled_entities(splits["train"], labels_by_entity, fraction, seed)
            run_id = f"{baseline_name}_f{int(fraction * 100):03d}_s{seed}"
            run_dir = CKPT_DIR / run_id
            print(
                f"mode={mode} fraction={fraction:.2f} seed={seed} "
                f"train_entities={len(train_ids)}"
            )

            model, best_val_auc = train_coles_classifier(mode, train_ids, seed, run_dir)
            test_metrics, pred_df = evaluate_coles_classifier(model, test_loader)
            row = {
                "baseline": baseline_name,
                "mode": mode,
                "fraction": fraction,
                "seed": seed,
                "best_val_roc_auc": float(best_val_auc),
                **test_metrics,
            }
            run_rows.append(row)
            print(row)

            pred_df.insert(0, GROUP_COL, splits["test"][: len(pred_df)])
            pred_df["baseline"] = baseline_name
            pred_df["mode"] = mode
            pred_df["fraction"] = fraction
            pred_df["seed"] = seed
            prediction_rows.append(pred_df)

            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

result_rows = []
metric_cols = ["best_val_roc_auc"] + METRIC_COLS
for mode in COLES_DOWNSTREAM_MODES:
    baseline_name = f"coles_{mode}"
    for fraction in LABEL_FRACTIONS:
        runs = [
            r for r in run_rows
            if r["baseline"] == baseline_name and float(r["fraction"]) == float(fraction)
        ]
        result_rows.extend(runs)
        mean_row = {
            "baseline": baseline_name,
            "mode": mode,
            "fraction": fraction,
            "seed": "mean",
            "n_seeds": len(runs),
        }
        std_row = {
            "baseline": baseline_name,
            "mode": mode,
            "fraction": fraction,
            "seed": "std",
            "n_seeds": len(runs),
        }
        for metric in metric_cols:
            vals = [float(r[metric]) for r in runs if metric in r and pd.notna(r[metric])]
            if vals:
                mean_row[metric] = float(np.mean(vals))
                std_row[metric] = float(np.std(vals, ddof=0))
        result_rows.append(mean_row)
        result_rows.append(std_row)

results_df = save_results(result_rows, "results_coles_ptls")

predictions_df = pd.concat(prediction_rows, ignore_index=True)
pred_path = OUTPUT_DIR / "predictions_coles_ptls.csv"
predictions_df.to_csv(pred_path, index=False)
print(f"Saved: {pred_path}")

try:
    display(results_df)
except NameError:
    print(results_df)
